In [7]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
!git clone https://github.com/mishitadudhaiya-code/NLP-Punctuation-Restoration-and-Translation

Cloning into 'NLP-Punctuation-Restoration-and-Translation'...
remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 6 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (6/6), done.


In [21]:
%cd NLP-Punctuation-Restoration-and-Translation
!git fetch origin
!git switch --track origin/task-1-cadence-punctuation

/content/NLP-Punctuation-Restoration-and-Translation/NLP-Punctuation-Restoration-and-Translation
branch 'task-1-cadence-punctuation' set up to track 'origin/task-1-cadence-punctuation'.
Switched to a new branch 'task-1-cadence-punctuation'


In [22]:
!git branch

  main
* task-1-cadence-punctuation


In [17]:


import os

PROJECT_DIR = "/content/drive/MyDrive/Research_Project"

MODEL1_DIR = os.path.join(PROJECT_DIR, "Model_1")

MODEL_DIR = os.path.join(MODEL1_DIR, "Cadence_Fast")
INPUT_DIR = os.path.join(MODEL1_DIR, "input_texts")
OUTPUT_DIR = os.path.join(MODEL1_DIR, "punctuation_outputs")

print("Model 1 paths ready!")
print("Model:", MODEL_DIR)
print("Input:", INPUT_DIR)
print("Output:", OUTPUT_DIR)



Model 1 paths ready!
Model: /content/drive/MyDrive/Research_Project/Model_1/Cadence_Fast
Input: /content/drive/MyDrive/Research_Project/Model_1/input_texts
Output: /content/drive/MyDrive/Research_Project/Model_1/punctuation_outputs


In [25]:
import torch

assert torch.cuda.is_available(), (
    "GPU not detected. In Colab, select Runtime > Change runtime type > T4 GPU."
)

print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory:",
      round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
      "GB")

GPU: Tesla T4
GPU memory: 14.56 GB


In [ ]:
!pip install -q --force-reinstall --no-deps transformers==4.57.1 tokenizers==0.22.1

In [41]:
import os
from pathlib import Path
from transformers import AutoTokenizer

# Find the tokenizer already downloaded in Google Drive
matches = list(Path(MODEL_DIR).rglob("tokenizer.json"))

print("Tokenizer files found:", len(matches))

if matches:
    TOKENIZER_DIR = str(matches[0].parent)
    print("Using local folder:", TOKENIZER_DIR)

    tokenizer = AutoTokenizer.from_pretrained(
        TOKENIZER_DIR,
        local_files_only=True
    )

    print("SUCCESS: Tokenizer loaded locally!")
else:
    print("Tokenizer file not found in MODEL_DIR.")

Tokenizer files found: 1
Using local folder: /content/drive/MyDrive/Research_Project/Model_1/Cadence_Fast/models--ai4bharat--Cadence-Fast/snapshots/8971c5011e4fba5dcfbcac52744587d7da605534
SUCCESS: Tokenizer loaded locally!


In [42]:
from transformers import AutoModel
import torch

model = AutoModel.from_pretrained(
    TOKENIZER_DIR,
    trust_remote_code=True,
    torch_dtype=torch.float32,
    local_files_only=True
)

model = model.to("cuda")
model.eval()

id2label = model.config.id2label

print("Cadence-Fast loaded successfully!")
print("Device:", next(model.parameters()).device)
print("Dtype:", next(model.parameters()).dtype)

`torch_dtype` is deprecated! Use `dtype` instead!
The following generation flags are not valid and may be ignored: ['cache_implementation']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Cadence-Fast loaded successfully!
Device: cuda:0
Dtype: torch.float32


In [6]:
PROJECT_DIR = "/content/drive/MyDrive/Research_Project"

INPUT_DIR = os.path.join(PROJECT_DIR, "input_texts")
OUTPUT_DIR = os.path.join(PROJECT_DIR, "punctuation_outputs")

print("Input folder exists:", os.path.exists(INPUT_DIR))
print("Output folder exists:", os.path.exists(OUTPUT_DIR))

files = sorted([
    f for f in os.listdir(INPUT_DIR)
    if f.lower().endswith(".txt")
])

print("\nAvailable input files:")
for i, f in enumerate(files):
    print(i, ":", f)

Input folder exists: True
Output folder exists: True

Available input files:


In [58]:

def punctuate_text_full(text, window_size=300, overlap=60):
    token_ids = tokenizer(
        text,
        add_special_tokens=False
    )["input_ids"]

    total = len(token_ids)
    predictions = [None] * total

    stride = window_size - overlap
    bos_id = tokenizer.bos_token_id

    for start in range(0, total, stride):
        end = min(start + window_size, total)
        chunk = token_ids[start:end]

        # Add BOS token to each window
        window_ids = [bos_id] + chunk

        input_ids = torch.tensor(
            [window_ids], device="cuda"
        )
        attention_mask = torch.ones_like(input_ids)

        with torch.inference_mode():
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        chunk_predictions = (
            outputs.logits[0].argmax(dim=-1).tolist()
        )

        # Ignore overlap edges except at document boundaries
        valid_start = 0 if start == 0 else overlap // 2
        valid_end = (
            len(chunk) if end == total
            else len(chunk) - overlap // 2
        )

        for j in range(valid_start, valid_end):
            predictions[start + j] = chunk_predictions[j + 1]

        del outputs, input_ids, attention_mask

        if end == total:
            break

    tokens = tokenizer.convert_ids_to_tokens(token_ids)
    result = []

    for token, pred in zip(tokens, predictions):
        if token == "\n":
            result.append("\n")
            continue

        result.append(token)

        punctuation = id2label[pred]
        if punctuation != "O":
            result.append(punctuation)

    return tokenizer.convert_tokens_to_string(result)


In [60]:
import os

for filename in files:
    input_path = os.path.join(INPUT_DIR, filename)

    with open(input_path, "r", encoding="utf-8") as f:
        original = f.read()

    result = punctuate_text_full(original)

    output_name = filename.replace(".txt", "_punctuated.txt")
    output_path = os.path.join(OUTPUT_DIR, output_name)

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(result)

    print(
        filename,
        "| Original:", len(original),
        "| Output:", len(result),
        "| Saved:", output_name
    )

print("\nAll seven files processed!")

sample-1.txt | Original: 3797 | Output: 3858 | Saved: sample-1_punctuated.txt
sample-2.txt | Original: 3825 | Output: 3887 | Saved: sample-2_punctuated.txt
sample-3.txt | Original: 3931 | Output: 3990 | Saved: sample-3_punctuated.txt
sample-4.txt | Original: 5428 | Output: 5532 | Saved: sample-4_punctuated.txt
sample-5.txt | Original: 4992 | Output: 5118 | Saved: sample-5_punctuated.txt
sample-6.txt | Original: 2330 | Output: 2375 | Saved: sample-6_punctuated.txt
sample-7.txt | Original: 3913 | Output: 3967 | Saved: sample-7_punctuated.txt

All seven files processed!


In [4]:
%cd NLP-Punctuation-Restoration-and-Translation

/content/NLP-Punctuation-Restoration-and-Translation


In [5]:
!git fetch origin
!git checkout task-1-cadence-punctuation
!git pull origin task-1-cadence-punctuation

branch 'task-1-cadence-punctuation' set up to track 'origin/task-1-cadence-punctuation'.
Switched to a new branch 'task-1-cadence-punctuation'
From https://github.com/mishitadudhaiya-code/NLP-Punctuation-Restoration-and-Translation
 * branch            task-1-cadence-punctuation -> FETCH_HEAD
Already up to date.


In [8]:
import os

PROJECT_DIR = "/content/drive/MyDrive/Research_Project"

print(os.listdir(PROJECT_DIR))

['Model_1', 'Cadence_Fast', 'input_texts', 'punctuation_outputs']


In [10]:
import os
import shutil

SOURCE = "/content/drive/MyDrive/Colab Notebooks/NLP.ipynb"
DESTINATION = "/content/NLP-Punctuation-Restoration-and-Translation/NLP.ipynb"

print("Source exists:", os.path.exists(SOURCE))
print("Destination folder exists:", os.path.isdir(os.path.dirname(DESTINATION)))

Source exists: True
Destination folder exists: True


In [11]:
shutil.copy(SOURCE, DESTINATION)
print("Task 1 notebook copied successfully!")

Task 1 notebook copied successfully!
